# verify05: 頭痛3質問を1つのBERTが学習（遷移なし・各質問の正答率）

verify04 の **B（遷移BERT）から 遷移・履歴・トリアージを外した版**。
**頭痛の3質問ノード（痛み/しびれ/異常言動）を1本のBERTがまとめて学習**し、質問ごとの正答率を見る。

### v2の変更点（非該当-1の精度改善＋内容語へのattention）
初版は入力が「ペア文だけ」だったため、**どの質問に対する判定かをモデルが知らず**、
`-1(非該当)`＝別質問のペア（例: しびれ質問）でも回答の「はい/いいえ」を読んで 0/1 と誤答し、
-1の正答率が **~0.31** に低迷。attentionも「はい/いいえ」語ばかりに向いた。

→ **入力を文ペア `(対象ノードの質問, ペア)` に変更**。「このペアは対象質問に答えているか？」を解かせる。
- -1判定に はい/いいえ は無力 → **内容語（激しく/突然/しびれ/麻痺）を見ざるを得ない** → attentionが内容へ。
- はい/いいえ/不明 も対象質問と照合して判定。

| 観点 | verify04 (B0/B1) | **verify05 v2** |
|---|---|---|
| モデル | 共有BERT + 遷移 | **共有BERT 1本（遷移なし）** |
| 入力 | 質問+採用ペア(+履歴) | **文ペア (対象ノードの質問, ペア)** |
| ラベル | はい/いいえ/不明(3値) | **はい/いいえ/不明/非該当（0/1/2/-1）** |
| 評価 | ノード別 + トリアージ | **各質問ノードの正答率のみ（トリアージなし）** |

- データ: `dataset/headache_symptom_pair_conversations_202607081836.csv`（3600ペア＝各ノード1200, 4ラベル×300で均衡）
- ラベル: `0=はい / 1=いいえ / 2=不明 / -1=非該当（その質問の答えになっていない別質問のペア）`
- 5-fold CV（node×label層化）で全ペアのOOF予測→質問ノードごとに正答率を集計。
- 学習ループ・`eval()`固定は verify04 / painful 流用。

# 1. セットアップ（GPUは git clone / ローカルはそのまま）

In [ ]:
import os, sys, subprocess, glob

REPO_URL = 'https://github.com/enenen13/Emergency_task'
REPO_NAME = 'Emergency_task'
REPO_BRANCH = 'feature/headache-ablation-notebook'
IN_COLAB = 'google.colab' in sys.modules

HEADACHE_CSV = 'headache_symptom_pair_conversations_202607081836.csv'


def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'dataset', HEADACHE_CSV)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_DIR = _find_repo_root(os.getcwd())
cloned = False
if REPO_DIR is None:
    if not os.path.isdir(REPO_NAME):
        print(f'git clone -b {REPO_BRANCH} {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        cloned = True
    REPO_DIR = os.path.abspath(REPO_NAME)
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

if IN_COLAB or cloned:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'transformers', 'sentencepiece', 'fugashi', 'unidic-lite',
                    'accelerate'], check=True)

CSV_PATH = os.path.join(REPO_DIR, 'dataset', HEADACHE_CSV)
OUT_DIR = os.path.join(REPO_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)
print('CSV :', CSV_PATH, '(exists:', os.path.exists(CSV_PATH), ')')

# 2. インポート & 設定

In [ ]:
import time, random
from typing import Dict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import transformers
transformers.logging.set_verbosity_error()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

# ===== 設定 =====
BASE_MODEL = 'cl-tohoku/bert-base-japanese-v3'   # 差し替えで別モデル可
MAX_LENGTH = 128            # 文ペア(質問+ペア)でも十分
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
N_FOLDS = 5
SEED = 42

INCLUDE_NONAPPLICABLE = True   # -1（非該当）を含めるか。Falseで はい/いいえ/不明 の3値のみ
SAMPLE_PER_NODE = None         # CPUで速く回すとき: 例 200。None=全件


def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print(f'BASE_MODEL={BASE_MODEL} MAX_LENGTH={MAX_LENGTH} epochs={NUM_EPOCHS} folds={N_FOLDS} '
      f'include_-1={INCLUDE_NONAPPLICABLE} sample/node={SAMPLE_PER_NODE}')

# 3. データ読み込み：文ペア (対象ノードの質問, ペア) を作る

各行が1ペア（相談員Q＋通報者A）。`is_*` でノード、`label_*` で正解。
**入力A＝対象ノードの代表質問（NODE_Q）**、**入力B＝ペア本文**。
-1（非該当）は「Bが別質問のペア」なので、A（対象質問）と照合すれば内容で弾ける。

In [ ]:
NODES = [
    {'key': 'sudden_severe', 'flag': 'is_sudden_severe', 'lab': 'label_sudden_severe', 'jp': '痛み（突然の激痛か）'},
    {'key': 'numbness',      'flag': 'is_numbness',      'lab': 'label_numbness',      'jp': 'しびれ／麻痺'},
    {'key': 'behavior',      'flag': 'is_behavior',      'lab': 'label_behavior',      'jp': '異常な言動・行動'},
]
JP = {n['key']: n['jp'] for n in NODES}
LABEL_MEANING = {'0': 'はい', '1': 'いいえ', '2': '不明', '-1': '非該当'}

# 各ノードの「対象質問」（文ペアの入力A）。データ中の代表質問を採用。
NODE_Q = {
    'sudden_severe': '頭の痛みは、突然起こった激しい痛みですか？',
    'numbness':      '手足のしびれや麻痺はありますか？',
    'behavior':      '普段と違う振る舞いや様子はありますか？',
}

d = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

def _node_of(r):
    for n in NODES:
        if bool(r[n['flag']]):
            return n['key']
    return None

_labcol = {n['key']: n['lab'] for n in NODES}
recs = []
for _, r in d.iterrows():
    nd = _node_of(r)
    if nd is None:
        continue
    code = str(r[_labcol[nd]])
    if (not INCLUDE_NONAPPLICABLE) and code == '-1':
        continue
    recs.append({'row_id': r['ID'], 'node': nd,
                 'prompt': NODE_Q[nd],           # 入力A: 対象質問
                 'text': str(r['ペア']),          # 入力B: ペア本文
                 'code': code})
ex = pd.DataFrame(recs)

if SAMPLE_PER_NODE:
    _n = int(min(SAMPLE_PER_NODE, ex.groupby('node').size().min()))
    ex = ex.groupby('node', group_keys=False).sample(n=_n, random_state=SEED).reset_index(drop=True)

le = LabelEncoder()
ex['label'] = le.fit_transform(ex['code'])
NUM_LABELS = len(le.classes_)
print('ペア数:', len(ex), '/ ノード:', ex['node'].nunique(), '/ ラベル:', list(le.classes_),
      '→ NUM_LABELS =', NUM_LABELS)
print('ノード×ラベル件数:')
display(pd.crosstab(ex['node'], ex['code']))
display(ex[['node', 'prompt', 'text', 'code']].head(4))

# 4. 学習・モデル部品（文ペア版）

`tokenizer(prompt, text)` で文ペア入力（segment埋め込みでA/Bを区別）。
学習ループ・`eval()`固定は verify04 流用。

In [ ]:
_tok_cache: Dict[str, 'AutoTokenizer'] = {}
def get_tokenizer(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    return _tok_cache[name]


class PairDataset(Dataset):
    # 文ペア (prompt=A, text=B) → ラベル
    def __init__(self, prompts, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.prompts, self.texts, self.labels = list(prompts), list(texts), list(labels)
        self.tokenizer, self.max_length = tokenizer, max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.prompts[idx], self.texts[idx], truncation=True,
                             max_length=self.max_length, padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def build_model(name):
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=NUM_LABELS,
                                                               trust_remote_code=True)
    tok = get_tokenizer(name)
    if len(tok) != model.config.vocab_size:
        print(f'  [警告] 語彙数{len(tok)}!=vocab{model.config.vocab_size} → resize')
        model.resize_token_embeddings(len(tok))
    return model.to(DEVICE)


def train_model(prompts, texts, labels):
    set_seed(SEED)
    tok = get_tokenizer(BASE_MODEL)
    model = build_model(BASE_MODEL)
    loader = DataLoader(PairDataset(prompts, texts, labels, tok, MAX_LENGTH),
                        batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    model.train()
    for epoch in range(NUM_EPOCHS):
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
    model.eval()   # ★推論前にevalへ（dropoutを切る）
    return model, tok


@torch.no_grad()
def predict_pairs(model, tok, prompts, texts):
    model.eval()
    prompts, texts = list(prompts), list(texts)
    preds = []
    for i in range(0, len(texts), EVAL_BATCH_SIZE):
        enc = tok(prompts[i:i + EVAL_BATCH_SIZE], texts[i:i + EVAL_BATCH_SIZE],
                  truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        logits = model(**enc).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())
    return preds


print('学習部品を定義（文ペア版・verify04流用・eval固定）')

# 5. 5-fold CV（node×label 層化）で全ペアのOOF予測

CPUの全件学習は重い（数十分〜）。速く見るなら設定セルで `SAMPLE_PER_NODE=200` か `N_FOLDS` を小さく。

In [ ]:
strat = (ex['node'].astype(str) + '_' + ex['code'].astype(str)).values
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

ex = ex.reset_index(drop=True)
ex['pred'] = -99
for fold, (tr, te) in enumerate(skf.split(ex.index, strat)):
    t0 = time.time()
    model, tok = train_model(ex.loc[tr, 'prompt'].tolist(), ex.loc[tr, 'text'].tolist(),
                             ex.loc[tr, 'label'].tolist())
    preds = predict_pairs(model, tok, ex.loc[te, 'prompt'].tolist(), ex.loc[te, 'text'].tolist())
    ex.loc[te, 'pred'] = preds
    print(f'  fold{fold}: train={len(tr)} test={len(te)}  ({time.time()-t0:.0f}s)')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert (ex['pred'] != -99).all(), 'OOF未充填の例がある'
print('OOF予測 完了')

# 6. 結果：各質問での正答率（トリアージなし）

In [ ]:
ex['correct'] = (ex['pred'] == ex['label']).astype(int)
overall_acc = ex['correct'].mean()
overall_f1 = f1_score(ex['label'], ex['pred'], average='macro', zero_division=0)
print(f'=== 全体 ===  accuracy={overall_acc:.3f}  macro-F1={overall_f1:.3f}  (n={len(ex)})')

node_tbl = (ex.groupby('node')
              .agg(n=('correct', 'size'), 正答率=('correct', 'mean'))
              .reset_index())
node_tbl.insert(1, '質問', node_tbl['node'].map(JP))
node_tbl['正答率'] = node_tbl['正答率'].map(lambda x: f'{x:.3f}')
print('\n===== 各質問ノードでの正答率 =====')
display(node_tbl)

ex['正解ラベル'] = ex['code'].map(LABEL_MEANING)
lab_tbl = (ex.groupby(['node', '正解ラベル'])
             .agg(n=('correct', 'size'), 正答率=('correct', 'mean'))
             .reset_index())
lab_tbl['正答率'] = lab_tbl['正答率'].map(lambda x: f'{x:.3f}')
print('\n===== ノード×正解ラベル別 正答率 =====')
display(lab_tbl)

node_tbl.to_csv(os.path.join(OUT_DIR, 'verify05_headache_node_acc.csv'), index=False, encoding='utf-8-sig')
lab_tbl.to_csv(os.path.join(OUT_DIR, 'verify05_headache_label_acc.csv'), index=False, encoding='utf-8-sig')
print('\nsaved: output/verify05_headache_node_acc.csv, output/verify05_headache_label_acc.csv')

# 7. Attention可視化（文ペア対応）

学習ループのモデルは `del` されるので、**attention出力を有効(eager)にした可視化用モデル**を1本だけ用意。
`(対象質問, ペア)` を入れ、CLS が各トークンにどれだけ注目したか（最終層・ヘッド平均）をHTMLで色付け。
文ペア化により、「はい/いいえ」語だけでなく **激しく/突然/しびれ 等の内容語** に注目が向くかを確認する。

In [ ]:
from IPython.display import display, HTML

VIZ_SAMPLE_PER_NODE = 200   # 可視化用モデルの学習量（CPU配慮）。None=全件


def build_viz_model(name):
    m = AutoModelForSequenceClassification.from_pretrained(
        name, num_labels=NUM_LABELS, trust_remote_code=True,
        attn_implementation='eager',   # ★ attentionを返すのに必須（既定sdpaだと出ない）
        output_attentions=True)
    t = get_tokenizer(name)
    if len(t) != m.config.vocab_size:
        m.resize_token_embeddings(len(t))
    return m.to(DEVICE)


_vz = ex
if VIZ_SAMPLE_PER_NODE:
    _n = int(min(VIZ_SAMPLE_PER_NODE, ex.groupby('node').size().min()))
    _vz = ex.groupby('node', group_keys=False).sample(n=_n, random_state=SEED).reset_index(drop=True)

set_seed(SEED)
viz_tok = get_tokenizer(BASE_MODEL)
viz_model = build_viz_model(BASE_MODEL)
_loader = DataLoader(PairDataset(_vz['prompt'].tolist(), _vz['text'].tolist(), _vz['label'].tolist(),
                                 viz_tok, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
_opt = torch.optim.AdamW(viz_model.parameters(), lr=LEARNING_RATE)
viz_model.train()
for _ep in range(NUM_EPOCHS):
    for _b in _loader:
        _b = {k: v.to(DEVICE) for k, v in _b.items()}
        _opt.zero_grad(); _o = viz_model(**_b); _o.loss.backward(); _opt.step()
viz_model.eval()
print('可視化用モデル 学習完了 (n=%d)' % len(_vz))


@torch.no_grad()
def show_attention(node_key, pair_text, layer=-1):
    # (対象質問, ペア) を入れ、CLS→各トークンのattention（最終層・ヘッド平均）を色付け表示
    prompt = NODE_Q[node_key]
    enc = viz_tok(prompt, pair_text, truncation=True, max_length=MAX_LENGTH,
                  return_tensors='pt', add_special_tokens=True).to(DEVICE)
    out = viz_model(**enc)
    pred_code = le.inverse_transform([int(out.logits.argmax(-1).cpu())])[0]
    pred_txt = LABEL_MEANING.get(str(pred_code), str(pred_code))

    attn = out.attentions[layer][0]        # [heads, seq, seq]
    cls = attn.mean(dim=0)[0]              # ヘッド平均 → CLS行
    tokens = viz_tok.convert_ids_to_tokens(enc['input_ids'][0])
    w = cls.clone(); w[0] = 0.0            # CLS自身は除外
    w = (w / (w.max() + 1e-9)).cpu().tolist()

    html = ''.join(
        f'<span style="background-color: rgba(255,80,80,{a:.3f}); padding:2px; margin:1px; '
        f'border-radius:3px;">{t}</span> ' for t, a in zip(tokens, w))
    display(HTML(f'<b>対象:{JP[node_key]} / 予測:{pred_txt}(code={pred_code})</b><br>'
                 f'<span style="color:#888">A={prompt}</span><br>{html}'))


# --- 使い方: (ノード, ペア文) を渡す ---
_r = ex[ex['node'] == 'sudden_severe'].iloc[0]
show_attention('sudden_severe', _r['text'])
# 例: -1（別質問のペア）を対象質問と照合 → 非該当を当てられるか
_neg = ex[(ex['node'] == 'sudden_severe') & (ex['code'] == '-1')]
if len(_neg):
    show_attention('sudden_severe', _neg.iloc[0]['text'])

# 8. まとめ（読み方）

- **文ペア化 (対象質問, ペア)** で、モデルは「このペアが対象質問に答えているか」を判定。
  - `-1(非該当)` は はい/いいえ では解けず内容照合が必須 → 内容語へのattention＆-1精度の改善を狙う。
- ノード×ラベル別の表で、特に **非該当(-1)** と **不明** の改善を初版(~0.31)と比較する。
- さらに上げるなら: (1) epochs↑ / lr調整、(2) 対象質問の言い回しを複数与える、(3) 長文モデルへ差し替え、(4) -1を「どのノードのペアか」まで当てる補助タスク。
- 遷移・履歴・トリアージは無し。各ペアは独立に「(質問, ペア) ⇒ ラベル」を当てるだけ。